In [32]:
import time              #用来暂停
import requests           #网上下载数据
import pandas as pd       #处理表格
companies = pd.read_csv("../data/raw/sp500_list.csv", dtype={"CIK": str})   #读取数据
companies.head()

,代码,公司,行业,加入日期,CIK,成立年份,成立年份_清洗
0,MMM,3M,Industrials,1957-03-04,0000066740,1902,1902
1,AOS,A. O. Smith,Industrials,2017-07-26,0000091142,1916,1916
2,ABT,Abbott Laboratories,Health Care,1957-03-04,0000001800,1888,1888
3,ABBV,AbbVie,Health Care,2012-12-31,0001551152,2013 (1888),2013
4,ACN,Accenture,Information Technology,2011-07-06,0001467373,1989,1989


In [33]:
# SEC 规定：每次请求都要写明"你是谁"（名字 + 邮箱），不写可能被拒绝
HEADERS = {"User-Agent": "Ziran Xing simonorbit04@gmail.com "}

# 要拿的会计科目
TAGS = [
    "NetIncomeLoss", "ProfitLoss",                                   # 净利润
    "NetCashProvidedByUsedInOperatingActivities",                    # 经营现金流
    "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",# 经营现金流（持续经营）
    "Assets",                                                        # 总资产
    "Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax",# 营收
]

In [34]:
def get_facts(cik):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

apple = get_facts("0000320193")
apple["facts"]["us-gaap"]["NetIncomeLoss"]["units"]["USD"][-3:] #取APPLE公司的NetIncomeloss试一试

[{'start': '2025-09-28',
  'end': '2026-06-27',
  'val': 101464000000,
  'accn': '0000320193-26-000020',
  'fy': 2026,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2026-07-31'},
 {'start': '2026-03-29',
  'end': '2026-06-27',
  'val': 29789000000,
  'accn': '0000320193-26-000020',
  'fy': 2026,
  'fp': 'Q3',
  'form': '10-Q',
  'filed': '2026-07-31',
  'frame': 'CY2026Q2'},
 {'start': '2025-09-28',
  'end': '2026-09-26',
  'val': 132175000000,
  'accn': '0000320193-26-000026',
  'fy': 2026,
  'fp': 'FY',
  'form': '10-K',
  'filed': '2026-11-03',
  'frame': 'CY2026'}]

In [35]:
rows= []   #数据篮子
failed=[]  #失败的公司
for i, c in companies.iterrows():
    try:
        facts = get_facts(c['CIK'])["facts"].get("us-gaap", {})
    except Exception as e:
        failed.append((c["代码"], str(e)))
        continue
    for tag in TAGS:
        if tag not in facts:
            continue
        for item in facts[tag]["units"].get("USD", []):
            if item.get("form") != "10-K":
                continue
            rows.append({"ticker" : c["代码"], 
                         "cik" : c['CIK'],
                         "sector" : c['行业'],  #从companies表格直接取出
                         
                         "tag" : tag,     #从TAG取出，区分不同的数值
                         
                         "start" : item.get("start"),
                         "end" : item["end"],
                         "value" : item["val"],
                         "fy" : item['fy'],
                         "fp" : item['fp'],
                         "form" : item['form'],
                         "filed" : item['filed'],
                         "frame" : item.get("frame")
                         })

    time.sleep(0.15)          # 每家停 0.15 秒，遵守 SEC 每秒 10 次的限制
    if i % 50 == 0:           # 每 50 家打印一次进度
        print(i, c["代码"])

0 MMM
50 ADSK
100 CMG
150 D
200 F
250 PODD
300 MLM
350 OKE
400 SPGI
450 TWLO
500 ZBRA


In [36]:
df = pd.DataFrame(rows)                           
df.to_csv("../data/raw/sec_raw.csv", index=False) 
print(df.shape)                                   
print("失败：", failed)

(141923, 12)
失败： []
